# AitW Single — 에피소드 종료 스텝 추출 & VLM 분석 파이프라인

## 플래그 조합

| `BACKEND` | `MODE` | 동작 |
|-----------|--------|------|
| `gemini`  | `single` | Gemini API 단일 요청 (토큰 사용량 출력) |
| `vllm`    | `single` | 로컬 vLLM 서버 단일 요청 (토큰 사용량 출력) |
| `gemini`  | `batch`  | Gemini Batch API 비동기 작업 제출 |
| `vllm`    | `batch`  | vLLM offline batch 실행 (CLI) |

**실행 순서**: Config 셀 → 유틸리티 셀 → 데이터 추출 셀 → 원하는 모드 셀

## 0. Config — 플래그 및 경로 설정

In [ ]:
# ── 플래그 (여기만 수정하면 됨) ────────────────────────────────────────
BACKEND = "gemini"   # "gemini" | "vllm"
MODE    = "batch"    # "single" | "batch"

# ── 서브셋 선택 ────────────────────────────────────────────────────────
# "general" | "google_apps" | "install" | "web_shopping" | "single"
SUBSET = "google_apps"

# 각 서브셋의 총 shard 수 (파일명 -of-XXXXX 값)
SUBSET_SHARD_COUNTS = {
    "general":      321,
    "google_apps":  8688,
    "install":      1052,
    "web_shopping": 1025,
    "single":       252,
}

# 서브셋별 권장 SHARD_INDICES (이미 다운로드된 것에 맞춰서 조정)
SHARD_INDICES_PRESETS = {
    "single":      [1, 2, 3, 4],            # 00000은 빈 파일(21 B)
    "google_apps": list(range(0, 11)),      # 0 ~ 10
    "general":      [],
    "install":      [],
    "web_shopping": [],
}

# 처리할 shard 인덱스 — preset 사용하거나 직접 지정
SHARD_INDICES = SHARD_INDICES_PRESETS[SUBSET]
# SHARD_INDICES = [1]  # 디버깅용 단일 샤드 override

# ── 경로 (SUBSET 기반 자동 생성) ──────────────────────────────────────
TOTAL_SHARDS = SUBSET_SHARD_COUNTS[SUBSET]
DATA_ROOT    = f"data/{SUBSET}"
EXTRACT_DIR  = f"data/{SUBSET}/extracted"

# ── 이미지 추출 설정 ───────────────────────────────────────────────────
# IMAGE_DOWNSCALE: 정수 축소 배수 (nearest neighbor 픽셀 추출)
#   1 = 축소 없음 (원본 해상도)  ← 기본값
#   2 = 가로/세로 1/2  (픽셀 수 1/4)
#   3 = 가로/세로 1/3  (픽셀 수 1/9)
IMAGE_DOWNSCALE   = 1
IMAGE_WEBP_QUALITY = 90   # 1-100, 높을수록 화질 ↑ 파일크기 ↑

# ── Gemini 설정 ────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
GEMINI_MODEL   = "gemini-3.1-flash-lite"

# Thinking 설정 — 모델 세대에 따라 다른 파라미터 사용:
#   - Gemini 3:   thinking_level (enum) → "minimal" | "low" | "medium" | "high"
#   - Gemini 2.5: thinking_budget (int) → 0=비활성, 512=flash-lite 최소, -1=동적
GEMINI_THINKING_LEVEL  = "minimal"   # Gemini 3 전용
GEMINI_THINKING_BUDGET = None        # Gemini 2.5 사용 시 0 또는 512 등으로 설정

# ── vLLM 설정 ──────────────────────────────────────────────────────────
VLLM_BASE_URL = "http://localhost:8000/v1"
VLLM_MODEL    = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── Structured Output 스키마 ───────────────────────────────────────────
OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "app_name": {
            "type": "string",
            "description": "Name of the app used (e.g., Chrome, Gmail, eBay, Settings)",
        },
        "app_category": {
            "type": "string",
            "description": "Category of the app (e.g., Browser, Email, Shopping, Settings, Social)",
        },
    },
    "required": ["app_name", "app_category"],
}

# ── 분석 프롬프트 ─────────────────────────────────────────────────────
ANALYSIS_PROMPT = """Instruction: {goal}

This is the final screenshot of an Android task at the moment the task was declared complete.
Identify which app was used to complete this task.
Return ONLY a JSON object — no markdown, no explanation.
"""

print(f"BACKEND={BACKEND}  MODE={MODE}  SUBSET={SUBSET}")
print(f"  TOTAL_SHARDS    : {TOTAL_SHARDS}")
print(f"  SHARD_INDICES   : {SHARD_INDICES}")
print(f"  DATA_ROOT       : {DATA_ROOT}")
print(f"  EXTRACT_DIR     : {EXTRACT_DIR}")
print(f"  IMAGE_DOWNSCALE : {IMAGE_DOWNSCALE}x  (quality={IMAGE_WEBP_QUALITY})")

## 1. 공통 유틸리티

In [ ]:
# !uv pip install tensorflow pillow tqdm google-genai openai requests pyvips

import tensorflow as tf
import numpy as np
import json
import base64
import io
import time
from pathlib import Path
from PIL import Image as PILImage
from collections import defaultdict

# ── TFRecord feature spec ──────────────────────────────────────────────
FEATURE_SPEC = {
    "android_api_level": tf.io.FixedLenFeature([], tf.int64),
    "current_activity":  tf.io.FixedLenFeature([], tf.string),
    "device_type":       tf.io.FixedLenFeature([], tf.string),
    "episode_id":        tf.io.FixedLenFeature([], tf.string),
    "step_id":           tf.io.FixedLenFeature([], tf.int64),
    "episode_length":    tf.io.FixedLenFeature([], tf.int64),
    "goal_info":         tf.io.FixedLenFeature([], tf.string),
    "image/encoded":     tf.io.FixedLenFeature([], tf.string),
    "image/height":      tf.io.FixedLenFeature([], tf.int64),
    "image/width":       tf.io.FixedLenFeature([], tf.int64),
    "image/channels":    tf.io.FixedLenFeature([], tf.int64, default_value=3),
    "image/ui_annotations_text":      tf.io.VarLenFeature(tf.string),
    "image/ui_annotations_positions": tf.io.VarLenFeature(tf.float32),
    "image/ui_annotations_ui_types":  tf.io.VarLenFeature(tf.string),
    "results/action_type":  tf.io.FixedLenFeature([], tf.int64),
    "results/type_action":  tf.io.FixedLenFeature([], tf.string, default_value=b""),
    "results/yx_touch": tf.io.FixedLenFeature([2], tf.float32, default_value=[0., 0.]),
    "results/yx_lift":  tf.io.FixedLenFeature([2], tf.float32, default_value=[0., 0.]),
}

STATUS_COMPLETE   = 10
STATUS_IMPOSSIBLE = 11

# ── 이미지 저장 백엔드 감지 (pyvips → OpenCV → PIL 순) ────────────────
# pyvips는 ImportError 외에 libvips DLL 누락 시 OSError도 발생
try:
    import pyvips
    _IMG_BACKEND = "pyvips"
except (ImportError, OSError):
    try:
        import cv2 as _cv2
        _IMG_BACKEND = "opencv"
    except ImportError:
        _IMG_BACKEND = "pillow"

print(f"이미지 저장 백엔드: {_IMG_BACKEND}")

# ── 이미지 헬퍼 ───────────────────────────────────────────────────────
def decode_image(ex) -> np.ndarray:
    H = int(ex["image/height"].numpy())
    W = int(ex["image/width"].numpy())
    C = int(ex["image/channels"].numpy())
    return tf.io.decode_raw(ex["image/encoded"], tf.uint8).numpy().reshape(H, W, C)

def save_image_webp(arr: np.ndarray, path: str,
                    downscale: int = 1, quality: int = 90) -> None:
    """RGB numpy 배열을 WebP로 저장.

    downscale: 정수 축소 배수 (1=원본, 2=1/2, ...). nearest neighbor 픽셀 추출
               방식이라 텍스트가 흐려지지 않음.
    quality:   WebP 품질 (1-100).
    """
    if _IMG_BACKEND == "pyvips":
        vimg = pyvips.Image.new_from_array(arr)
        if downscale > 1:
            vimg = vimg.subsample(downscale, downscale)
        vimg.webpsave(path, Q=quality)
    elif _IMG_BACKEND == "opencv":
        import cv2
        bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
        if downscale > 1:
            h, w = bgr.shape[:2]
            bgr = cv2.resize(bgr, (w // downscale, h // downscale),
                             interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(path, bgr, [cv2.IMWRITE_WEBP_QUALITY, quality])
    else:
        small = arr[::downscale, ::downscale] if downscale > 1 else arr
        PILImage.fromarray(small).save(path, format="WEBP", quality=quality)

def file_to_bytes(path: str) -> bytes:
    with open(path, "rb") as f:
        return f.read()

def file_to_base64(path: str) -> str:
    return base64.b64encode(file_to_bytes(path)).decode()

print("유틸리티 로드 완료")

## 2. 데이터 추출 — 에피소드 종료 스텝 (STATUS_COMPLETE)

각 에피소드의 마지막 step (`action_type=10, STATUS_COMPLETE`) 이미지와 지시문을 추출해 저장합니다.

출력 구조:
```
data/single/extracted/
  shard00001_ep0000.png   ← 스크린샷
  shard00001_ep0000.json  ← 메타데이터
  index.jsonl             ← 전체 인덱스
```

In [ ]:
try:
    from tqdm import tqdm
except ImportError:
    from tqdm import tqdm

MIN_SHARD_SIZE = 1024  # 1 KB 미만은 빈 샤드로 간주 (예: single-00000 = 21 B)

Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)
index = []
skipped_shards = []   # 로컬 없음
empty_shards   = []   # 파일 있지만 빈/너무 작음
broken_shards  = []   # 읽기 중 오류

for shard_idx in SHARD_INDICES:
    shard_path = f"{DATA_ROOT}/{SUBSET}-{shard_idx:05d}-of-{TOTAL_SHARDS:05d}"

    if not Path(shard_path).exists():
        print(f"[skip] 로컬 없음: {shard_path}")
        skipped_shards.append(shard_idx)
        continue

    size = Path(shard_path).stat().st_size
    if size < MIN_SHARD_SIZE:
        print(f"[skip] 빈 샤드 ({size} B): {shard_path}")
        empty_shards.append(shard_idx)
        continue

    raw    = tf.data.TFRecordDataset([shard_path], compression_type="GZIP")
    parsed = raw.map(lambda x: tf.io.parse_single_example(x, FEATURE_SPEC))

    # 에피소드별 그룹핑 — 손상된 GZIP/TFRecord는 iteration 중 오류 발생 가능
    episodes = defaultdict(list)
    try:
        for ex in parsed:
            episodes[ex["episode_id"].numpy().decode()].append(ex)
    except (tf.errors.DataLossError, tf.errors.InvalidArgumentError) as e:
        print(f"[skip] 손상된 샤드 ({type(e).__name__}): {shard_path}")
        broken_shards.append(shard_idx)
        continue

    if not episodes:
        print(f"[skip] 에피소드 0개: {shard_path}")
        empty_shards.append(shard_idx)
        continue

    shard_count = 0
    for ep_idx, (eid, steps) in enumerate(
        tqdm(episodes.items(), desc=f"shard {shard_idx:05d}", leave=False)
    ):
        # STATUS_COMPLETE step 추출 (에피소드당 정확히 1개)
        complete = [
            s for s in steps
            if int(s["results/action_type"].numpy()) == STATUS_COMPLETE
        ]
        if not complete:
            continue
        ex = complete[0]

        stem      = f"shard{shard_idx:05d}_ep{ep_idx:04d}"
        img_name  = f"{stem}.webp"
        img_path  = f"{EXTRACT_DIR}/{img_name}"
        meta_path = f"{EXTRACT_DIR}/{stem}.json"

        # 이미지 저장 (IMAGE_DOWNSCALE 배수 축소, WebP)
        save_image_webp(
            decode_image(ex), img_path,
            downscale=IMAGE_DOWNSCALE, quality=IMAGE_WEBP_QUALITY,
        )

        # 메타데이터 저장
        meta = {
            "image":             img_name,
            "goal_info":         ex["goal_info"].numpy().decode(errors="replace"),
            "episode_id":        eid,
            "shard":             shard_idx,
            "ep_idx":            ep_idx,
            "step_id":           int(ex["step_id"].numpy()),
            "episode_length":    int(ex["episode_length"].numpy()),
            "device_type":       ex["device_type"].numpy().decode(),
            "android_api_level": int(ex["android_api_level"].numpy()),
            "current_activity":  ex["current_activity"].numpy().decode(errors="replace"),
        }
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

        index.append(meta)
        shard_count += 1

    print(f"  shard {shard_idx:05d}: {len(episodes)} 에피소드 → {shard_count}개 추출")

# 전체 인덱스 저장
index_path = f"{EXTRACT_DIR}/index.jsonl"
with open(index_path, "w", encoding="utf-8") as f:
    for item in index:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n추출 완료: {len(index)}개 image-instruction 쌍 → {index_path}")
if skipped_shards: print(f"스킵 (로컬 없음): {skipped_shards}")
if empty_shards:   print(f"스킵 (빈 샤드)  : {empty_shards}")
if broken_shards:  print(f"스킵 (손상)     : {broken_shards}")

## 3. 단일 요청 테스트 (MODE = "single")

index.jsonl 첫 번째 항목으로 테스트 요청을 보내고 응답과 토큰 사용량을 출력합니다.

In [ ]:
if MODE != "single":
    print(f"MODE='{MODE}' → 이 셀은 MODE='single'일 때만 실행됩니다. 건너뜁니다.")
else:
    index_path = f"{EXTRACT_DIR}/index.jsonl"
    with open(index_path, encoding="utf-8") as f:
        sample = json.loads(f.readline())

    single_img_path = f"{EXTRACT_DIR}/{sample['image']}"
    single_goal     = sample["goal_info"]
    single_prompt   = ANALYSIS_PROMPT.format(goal=single_goal)

    print("── 테스트 샘플 ──────────────────────────────────────────")
    print(f"image      : {single_img_path}")
    print(f"goal_info  : {single_goal}")
    print(f"device     : {sample['device_type']}  api_level={sample['android_api_level']}")
    print(f"episode_id : {sample['episode_id'][:80]}...")

In [ ]:
# ── Gemini 단일 요청 (structured output + thinking_level=minimal + media_resolution=LOW) ──
if MODE != "single" or BACKEND != "gemini":
    print(f"BACKEND='{BACKEND}', MODE='{MODE}' → Gemini 단일 요청 셀 건너뜀")
else:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt = ANALYSIS_PROMPT.format(goal=single_goal)
    img_bytes = file_to_bytes(single_img_path)
    print(f"이미지 원본 크기: {len(img_bytes) / 1024:.1f} KB → API 측 LOW 모드로 처리")

    # ThinkingConfig: Gemini 3는 thinking_level, Gemini 2.5는 thinking_budget 사용
    if GEMINI_THINKING_LEVEL is not None:
        thinking_cfg = types.ThinkingConfig(thinking_level=GEMINI_THINKING_LEVEL.upper())
        print(f"thinking_level={GEMINI_THINKING_LEVEL.upper()}")
    else:
        thinking_cfg = types.ThinkingConfig(thinking_budget=GEMINI_THINKING_BUDGET)
        print(f"thinking_budget={GEMINI_THINKING_BUDGET}")

    t0 = time.time()
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            types.Part.from_text(text=prompt),
            types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=OUTPUT_SCHEMA,
            thinking_config=thinking_cfg,
            media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW,
        ),
    )
    elapsed = time.time() - t0

    parsed = json.loads(resp.text)
    print(f"\n── Gemini ({GEMINI_MODEL}) 응답 ─────────────────────────")
    print(f"  APP     : {parsed.get('app_name', 'N/A')}")
    print(f"  CATEGORY: {parsed.get('app_category', 'N/A')}")
    print(f"\n  raw JSON: {resp.text}")

    usage = resp.usage_metadata
    thinking_tokens = getattr(usage, "thoughts_token_count", "N/A")
    print("\n── Token 사용량 ────────────────────────────────────────")
    print(f"  input (prompt)    : {usage.prompt_token_count:,} tokens")
    print(f"  thinking          : {thinking_tokens} tokens")
    print(f"  output (generated): {usage.candidates_token_count:,} tokens")
    print(f"  total             : {usage.total_token_count:,} tokens")
    print(f"  응답 시간         : {elapsed:.2f}s")

In [ ]:
# ── vLLM 단일 요청 (structured output via guided_json, detail=low) ────
# 사전 조건: vLLM 서버가 VLLM_BASE_URL에서 실행 중이어야 함
# 실행 예시: vllm serve Qwen/Qwen2.5-VL-7B-Instruct --port 8000
if MODE != "single" or BACKEND != "vllm":
    print(f"BACKEND='{BACKEND}', MODE='{MODE}' → vLLM 단일 요청 셀 건너뜀")
else:
    from openai import OpenAI

    client  = OpenAI(base_url=VLLM_BASE_URL, api_key="EMPTY")
    img_b64 = file_to_base64(single_img_path)
    prompt  = ANALYSIS_PROMPT.format(goal=single_goal)

    t0 = time.time()
    resp = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {
                    "url":    f"data:image/png;base64,{img_b64}",
                    "detail": "low",
                }},
                {"type": "text", "text": prompt},
            ],
        }],
        extra_body={"guided_json": OUTPUT_SCHEMA},
    )
    elapsed = time.time() - t0

    content = resp.choices[0].message.content
    parsed  = json.loads(content)

    print(f"\n── vLLM ({VLLM_MODEL}) 응답 ──────────────────────────")
    print(f"  APP     : {parsed.get('app_name', 'N/A')}")
    print(f"  CATEGORY: {parsed.get('app_category', 'N/A')}")
    print(f"\n  raw JSON: {content}")

    u = resp.usage
    print("\n── Token 사용량 ────────────────────────────────────────")
    print(f"  input (prompt)    : {u.prompt_tokens:,} tokens")
    print(f"  output (generated): {u.completion_tokens:,} tokens")
    print(f"  total             : {u.total_tokens:,} tokens")
    print(f"  응답 시간         : {elapsed:.2f}s")

## 4. 배치 요청 (MODE = "batch")

### 4-1. 배치 입력 JSONL 생성 (Gemini / vLLM 공통)

In [ ]:
if MODE != "batch":
    print(f"MODE='{MODE}' → 배치 셀은 MODE='batch'일 때만 실행됩니다.")
else:
    index_path = f"{EXTRACT_DIR}/index.jsonl"
    with open(index_path, encoding="utf-8") as f:
        index_data = [json.loads(line) for line in f]

    # 현재 SHARD_INDICES에 해당하는 항목만 필터 (디버깅용)
    filtered = [item for item in index_data if item["shard"] in SHARD_INDICES]
    print(f"전체 인덱스 {len(index_data)}건 → SHARD_INDICES={SHARD_INDICES} 필터: {len(filtered)}건")
    index_data = filtered

    batch_input_path  = f"{EXTRACT_DIR}/batch_input_{BACKEND}.jsonl"
    batch_output_path = f"{EXTRACT_DIR}/batch_output_{BACKEND}.jsonl"

    # Gemini thinking 필드 — Gemini 3는 thinkingLevel, Gemini 2.5는 thinkingBudget
    if GEMINI_THINKING_LEVEL is not None:
        thinking_field = {"thinkingLevel": GEMINI_THINKING_LEVEL.upper()}
    else:
        thinking_field = {"thinkingBudget": GEMINI_THINKING_BUDGET}

    with open(batch_input_path, "w", encoding="utf-8") as out:
        for item in index_data:
            img_path = f"{EXTRACT_DIR}/{item['image']}"
            goal     = item["goal_info"]
            prompt   = ANALYSIS_PROMPT.format(goal=goal)
            img_b64  = file_to_base64(img_path)
            req_id   = f"shard{item['shard']:05d}_ep{item['ep_idx']:04d}"

            if BACKEND == "gemini":
                req = {
                    "key": req_id,
                    "request": {
                        "contents": [{
                            "parts": [
                                {"text": prompt},
                                {"inline_data": {"mime_type": "image/png", "data": img_b64}},
                            ]
                        }],
                        "generationConfig": {
                            "maxOutputTokens":  256,
                            "responseMimeType": "application/json",
                            "responseSchema":   OUTPUT_SCHEMA,
                            "thinkingConfig":   thinking_field,
                            "mediaResolution":  "MEDIA_RESOLUTION_LOW",
                        },
                    },
                }
            elif BACKEND == "vllm":
                req = {
                    "custom_id": req_id,
                    "method":    "POST",
                    "url":       "/v1/chat/completions",
                    "body": {
                        "model": VLLM_MODEL,
                        "messages": [{
                            "role": "user",
                            "content": [
                                {"type": "image_url", "image_url": {
                                    "url":    f"data:image/png;base64,{img_b64}",
                                    "detail": "low",
                                }},
                                {"type": "text", "text": prompt},
                            ],
                        }],
                        "max_tokens":  256,
                        "guided_json": OUTPUT_SCHEMA,
                    },
                }
            out.write(json.dumps(req, ensure_ascii=False) + "\n")

    size_mb = Path(batch_input_path).stat().st_size / 1e6
    print(f"배치 JSONL 생성 완료: {batch_input_path}")
    print(f"  건수: {len(index_data)}개  크기: {size_mb:.1f} MB  백엔드: {BACKEND}  해상도: LOW")
    if BACKEND == "gemini":
        print(f"  thinking: {thinking_field}")

### 4-2. Gemini Batch API 제출 (BACKEND = "gemini")

In [ ]:
# Gemini Batch API — google-genai SDK 사용
# 참고: https://ai.google.dev/gemini-api/docs/batch-api
#
# Flow: files.upload → batches.create → batches.get (polling) → files.download
if MODE != "batch" or BACKEND != "gemini":
    print(f"BACKEND='{BACKEND}', MODE='{MODE}' → Gemini 배치 셀 건너뜀")
else:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=GEMINI_API_KEY)

    # ── Step 1: JSONL 파일 업로드 ─────────────────────────────────────
    print("[1/4] JSONL 파일 업로드 중...")
    uploaded_file = client.files.upload(
        file=batch_input_path,
        config=types.UploadFileConfig(
            mime_type="application/jsonl",
            display_name=f"aitw_batch_{BACKEND}",
        ),
    )
    print(f"  업로드 완료: {uploaded_file.name}  ({uploaded_file.size_bytes / 1e6:.1f} MB)")

    # ── Step 2: 배치 작업 생성 ────────────────────────────────────────
    print("[2/4] 배치 작업 생성 중...")
    batch_job = client.batches.create(
        model=GEMINI_MODEL,
        src=uploaded_file.name,
        config=types.CreateBatchJobConfig(display_name=f"aitw_batch_{BACKEND}"),
    )
    print(f"  배치 작업: {batch_job.name}  초기 상태: {batch_job.state.name}")

    # ── Step 3: 상태 폴링 ────────────────────────────────────────────
    print("[3/4] 상태 폴링 중... (30초 간격)")
    terminal_states = {
        "JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED",
        "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED",
    }
    while batch_job.state.name not in terminal_states:
        time.sleep(30)
        batch_job = client.batches.get(name=batch_job.name)
        print(f"  상태: {batch_job.state.name}")

    state = batch_job.state.name

    # ── Step 4: 결과 다운로드 ─────────────────────────────────────────
    print("[4/4] 결과 처리 중...")
    if state == "JOB_STATE_SUCCEEDED":
        dest = batch_job.dest
        if dest and getattr(dest, "file_name", None):
            # 파일 기반 결과
            result_bytes = client.files.download(file=dest.file_name)
            with open(batch_output_path, "wb") as f:
                f.write(result_bytes)
            print(f"  결과 저장: {batch_output_path}  ({len(result_bytes) / 1e3:.1f} KB)")
        elif dest and getattr(dest, "inlined_responses", None):
            # 인라인 결과 (소규모 배치)
            with open(batch_output_path, "w", encoding="utf-8") as f:
                for resp in dest.inlined_responses:
                    f.write(json.dumps(resp.model_dump(exclude_none=True),
                                       ensure_ascii=False) + "\n")
            print(f"  인라인 결과 저장: {batch_output_path}  ({len(dest.inlined_responses)}건)")
        else:
            print(f"  결과 형식 확인 필요: dest={dest}")
    else:
        print(f"  작업 실패: state={state}")
        if hasattr(batch_job, "error") and batch_job.error:
            print(f"  오류: {batch_job.error}")

### 4-3. vLLM Offline Batch 실행 (BACKEND = "vllm")

In [ ]:
# vLLM offline batch — 서버 없이 CLI로 직접 실행
# 참고: https://docs.vllm.ai/en/v0.9.0/examples/offline_inference/openai_batch.html
# 사전 조건: pip install vllm  /  GPU 환경 권장
if MODE != "batch" or BACKEND != "vllm":
    print(f"BACKEND='{BACKEND}', MODE='{MODE}' → vLLM 배치 셀 건너뜀")
else:
    import subprocess

    cmd = [
        "python", "-m", "vllm.entrypoints.openai.run_batch",
        "--input-file",  batch_input_path,
        "--output-file", batch_output_path,
        "--model",       VLLM_MODEL,
        "--max-model-len", "8192",
    ]
    print("실행 명령:")
    print(" ".join(cmd))
    print("─" * 60)

    result = subprocess.run(cmd, capture_output=False, text=True)

    if result.returncode != 0:
        print(f"\n[오류] returncode={result.returncode}")
    else:
        print(f"\n완료: {batch_output_path}")

## 5. 결과 파싱 및 저장

In [ ]:
if MODE != "batch":
    print(f"MODE='{MODE}' → 결과 파싱 셀은 MODE='batch'일 때만 실행됩니다.")
elif not Path(batch_output_path).exists():
    print(f"결과 파일 없음: {batch_output_path}")
else:
    results = []
    errors  = []

    with open(batch_output_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)

            if BACKEND == "gemini":
                key   = item.get("key", "")
                error = item.get("error")
                if error:
                    errors.append({"key": key, "error": error})
                else:
                    raw_text = (
                        item.get("response", {})
                            .get("candidates", [{}])[0]
                            .get("content", {})
                            .get("parts", [{}])[0]
                            .get("text", "")
                    )
                    try:
                        parsed = json.loads(raw_text)
                    except json.JSONDecodeError:
                        parsed = {"app_name": raw_text, "app_category": ""}
                    results.append({"key": key, **parsed})

            elif BACKEND == "vllm":
                cid   = item.get("custom_id", "")
                error = item.get("error")
                if error:
                    errors.append({"key": cid, "error": error})
                else:
                    body     = item.get("response", {}).get("body", {})
                    raw_text = body.get("choices", [{}])[0].get("message", {}).get("content", "")
                    usage    = body.get("usage", {})
                    try:
                        parsed = json.loads(raw_text)
                    except json.JSONDecodeError:
                        parsed = {"app_name": raw_text, "app_category": ""}
                    results.append({"key": cid, **parsed, "usage": usage})

    # Gemini Batch API는 응답 순서를 보장하지 않으므로 key 기준 정렬
    results.sort(key=lambda r: r["key"])
    errors.sort(key=lambda e: e["key"])

    # 결과 저장
    parsed_path = f"{EXTRACT_DIR}/results_{BACKEND}.json"
    with open(parsed_path, "w", encoding="utf-8") as f:
        json.dump({"results": results, "errors": errors}, f,
                  ensure_ascii=False, indent=2)

    print(f"파싱 완료: 성공 {len(results)}건 / 오류 {len(errors)}건 (key 정렬됨)")
    print(f"저장: {parsed_path}")

    print("\n── 샘플 결과 (최대 5건) ────────────────────────────────")
    for r in results[:5]:
        print(f"  [{r['key']}]  APP: {r.get('app_name','?')}  CATEGORY: {r.get('app_category','?')}")

    if errors:
        print(f"\n── 오류 샘플 ({len(errors)}건) ──────────────────────────")
        for e in errors[:3]:
            print(e)

In [ ]:
# ── 배치 결과 전체 출력 + 토큰 사용량 집계 ───────────────────────────
if not Path(batch_output_path).exists():
    print(f"결과 파일 없음: {batch_output_path}")
elif BACKEND != "gemini":
    print(f"BACKEND='{BACKEND}' → Gemini 배치 결과 출력 셀 건너뜀")
else:
    total_text_in   = 0
    total_image_in  = 0
    total_other_in  = 0
    total_prompt    = 0
    total_thinking  = 0
    total_output    = 0
    total_all       = 0
    n_items         = 0
    n_errors        = 0

    print("=" * 80)
    print(f"배치 결과 전체 내용: {batch_output_path}")
    print("=" * 80)

    with open(batch_output_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item    = json.loads(line)
            n_items += 1
            key     = item.get("key", "?")
            error   = item.get("error")

            print(f"\n── [{n_items}] key={key} " + "─" * (60 - len(key)))

            if error:
                n_errors += 1
                print(f"  ERROR: {error}")
                continue

            response = item.get("response", {})

            # 응답 텍스트 추출 & 출력
            raw_text = (
                response.get("candidates", [{}])[0]
                        .get("content", {})
                        .get("parts", [{}])[0]
                        .get("text", "")
            )
            try:
                parsed = json.loads(raw_text)
                print(f"  APP     : {parsed.get('app_name', 'N/A')}")
                print(f"  CATEGORY: {parsed.get('app_category', 'N/A')}")
            except json.JSONDecodeError:
                print(f"  RAW: {raw_text}")

            # 토큰 사용량 집계
            usage             = response.get("usageMetadata", {})
            prompt_tokens     = usage.get("promptTokenCount",     0)
            candidates_tokens = usage.get("candidatesTokenCount", 0)
            thoughts_tokens   = usage.get("thoughtsTokenCount",   0)
            total_tokens      = usage.get("totalTokenCount",      0)

            # 입력 모달리티별 분리
            text_tok = img_tok = other_tok = 0
            for d in usage.get("promptTokensDetails", []):
                mod = d.get("modality", "")
                cnt = d.get("tokenCount", 0)
                if   mod == "TEXT":  text_tok  += cnt
                elif mod == "IMAGE": img_tok   += cnt
                else:                other_tok += cnt

            total_text_in  += text_tok
            total_image_in += img_tok
            total_other_in += other_tok
            total_prompt   += prompt_tokens
            total_thinking += thoughts_tokens
            total_output   += candidates_tokens
            total_all      += total_tokens

            print(f"  tokens  : in={prompt_tokens} (text={text_tok}, img={img_tok})"
                  f"  thinking={thoughts_tokens}  out={candidates_tokens}  total={total_tokens}")

    # ── 전체 합계 ─────────────────────────────────────────────────────
    print(f"\n{'=' * 80}")
    print(f"전체 토큰 사용량 합계  ({n_items}건, 오류 {n_errors}건)")
    print(f"{'=' * 80}")
    print("입력 (input):")
    print(f"  텍스트         : {total_text_in:>10,} tokens")
    print(f"  이미지         : {total_image_in:>10,} tokens")
    if total_other_in > 0:
        print(f"  기타           : {total_other_in:>10,} tokens")
    print(f"  ───────────────────────────────")
    print(f"  입력 합계      : {total_prompt:>10,} tokens")
    print()
    print("출력 (output):")
    print(f"  thinking       : {total_thinking:>10,} tokens")
    print(f"  결과 텍스트    : {total_output:>10,} tokens")
    print(f"  ───────────────────────────────")
    print(f"  출력 합계      : {total_thinking + total_output:>10,} tokens")
    print()
    print(f"총합 (입력+출력) : {total_all:>10,} tokens")